# THEMIS-A: FAC基底のGSM表示とθ・ξ

`THEMIS_BE_dispersion_relation_double_second.ipynb`のセル29–32を参照する。
FGM L2 `tha_fgl_gsm`（公称16 Hz）の100秒中心移動平均を背景磁場とし、
FAC-Zを背景磁場方向、FAC-Xを位置方向の磁場垂直成分、FAC-YをZ×Xと定義する。
既にGSM表示なので、AraseのDSI→J2000→GSM変換に相当する処理は不要。

Arase版と同じ16×11 inch、150 dpi、3段構成で、上段(a)–(c)にXY・XZ・YZ投影、
中段(d)にθ（青）、下段(e)にξ（オレンジ）を表示する。
GSMの表示ラベルはX・Y・Z、極座標の補助線は15°、数値ラベルは45°刻み。
θ・ξには補助目盛りと`grid(which='both', alpha=0.3)`、上段の実時刻に赤い縦破線を付ける。

既定の対象は2022-09-01 22:25–23:15 UTC。毎分00秒に最も近い実サンプルを選び、
サンプル間隔の中央値を超えて離れた点と、FAC未定義の時刻は保存しない。
θ・ξの縦軸は指定区間の有限値から決め、全フレームで固定する。
最後の保存セルを実行するまで画像の一括保存は行わない。

参照との差は欠損処理にある。FGMの時間ギャップをまたぐ移動平均と、
軌道の長い欠損・範囲外への補間を禁止する。追加の品質フラグによる選別は行わず、
CDFローダーの欠損処理、有限性、窓内の有効点数で判定する。

In [ ]:
import os
os.environ['SPEDAS_DATA_DIR'] = '/mnt/j/observation_data/'

import numpy as np
import xarray as xr
import pyspedas as psp
from tqdm.auto import tqdm

probe = 'a'
spacecraft_label = f'THEMIS-{probe.upper()}'
# 100秒窓の端点と軌道補間に余裕を持たせ、参照ノートブックと同じ範囲を読む。
time_range = ['2022-09-01/20:00:00', '2022-09-02/00:00:00']
range_start = np.datetime64('2022-09-01T22:25:00', 'ns')
range_end = np.datetime64('2022-09-01T23:15:00', 'ns')
background_time_sec = 100.
fgm_gap_threshold_sec = 0.25  # 参照側の16 Hz区間分割と同じ250 ms
orbit_max_gap_sec = None     # None: 軌道時刻間隔の中央値の2倍
angle_singularity_tol = 1e-10
coordinate_matrix_atol = 1e-6
no_update = True             # 既存キャッシュを使用。取得が必要ならFalseに変更する。
range_plot_style = 'polar'   # 'cartesian'も指定可能
range_minute_tolerance_sec = None

## GSM磁場と軌道の読み込み

必要なのはFGMとSTATEのみ。電場・粒子データは読み込まない。
FGMはnT、位置はkmとして配布されるが、基底・θ・ξは方向だけを使う。
読み込み不足時は古いtplot変数を流用せずエラーにする。

In [ ]:
fgm_name = f'th{probe}_fgl_gsm'
position_name = f'th{probe}_pos_gsm'
fgm_loaded = psp.projects.themis.fgm(
    trange=time_range, probe=probe, level='l2', coord='gsm',
    varnames=[fgm_name], get_support_data=True, no_update=no_update,
)
state_loaded = psp.projects.themis.state(
    trange=time_range, probe=probe, varnames=[position_name], no_update=no_update,
)
if fgm_name not in (fgm_loaded or []) or position_name not in (state_loaded or []):
    raise RuntimeError('FGMまたはSTATEが読み込めない。キャッシュとno_update設定を確認する。')


def prepare_gsm_vector(data, start, end, name):
    if data is None or set(data.dims) != {'time', 'v_dim'} or data.sizes['v_dim'] != 3:
        raise ValueError(f'{name}: (time, v_dim=3)のベクトルが必要。')
    data = data.transpose('time', 'v_dim')
    data = data.isel(time=~data.get_index('time').duplicated()).sortby('time')
    data = data.sel(time=slice(start, end)).astype(np.float64)
    if data.sizes['time'] < 2 or np.isnat(data.time.values).any():
        raise ValueError(f'{name}: 有効な時刻が不足している。')
    return data.assign_coords(v_dim=[0, 1, 2]).where(np.isfinite(data))


load_start, load_end = [np.datetime64(t.replace('/', 'T'), 'ns') for t in time_range]
B16_data_gsm = prepare_gsm_vector(psp.get_data(fgm_name, xarray=True), load_start, load_end, fgm_name)
da_THA_pos_gsm = prepare_gsm_vector(psp.get_data(position_name, xarray=True), load_start, load_end, position_name)
print('FGM samples:', B16_data_gsm.sizes['time'], '| STATE samples:', da_THA_pos_gsm.sizes['time'])

## 100秒背景磁場、FAC基底と変換行列

$\boldsymbol e_z=\boldsymbol B_0/|\boldsymbol B_0|$、
$\boldsymbol e_x\propto\hat{\boldsymbol r}-(\hat{\boldsymbol r}\cdot\boldsymbol e_z)\boldsymbol e_z$、
$\boldsymbol e_y=\boldsymbol e_z\times\boldsymbol e_x$。
参照と同じく、FAC基底の位置方向は軌道位置の単位ベクトルを補間して使う。
磁場ゼロ、動径と磁場が平行、軌道欠損、移動平均窓の不足はNaNとし、時刻列は保持する。
列ベクトル規約で`R_FAC_to_GSM(time, gsm_component, fac_component)`を作る。

In [ ]:
def interpolate_position_direction(position, target_time, max_gap_sec=None):
    times = position.time.values.astype('datetime64[ns]')
    targets = target_time.values.astype('datetime64[ns]')
    dt = np.diff(times) / np.timedelta64(1, 's')
    if np.any(dt <= 0):
        raise ValueError('軌道時刻は単調増加である必要がある。')
    if max_gap_sec is None:
        max_gap_sec = float(2 * np.median(dt))
    if not np.isfinite(max_gap_sec) or max_gap_sec <= 0:
        raise ValueError('軌道の最大補間間隔は正の有限値にする。')
    radius = np.linalg.norm(position.values, axis=1)
    unit = position / xr.DataArray(np.where(radius > 0, radius, np.nan), dims='time', coords={'time': position.time})
    result = unit.interp(time=target_time)
    right = np.searchsorted(times, targets)
    left = np.clip(right - 1, 0, len(times) - 1)
    clipped = np.clip(right, 0, len(times) - 1)
    exact = times[clipped] == targets
    bracket = (right > 0) & (right < len(times))
    bracket &= (times[clipped] - times[left]) / np.timedelta64(1, 's') <= max_gap_sec
    return result.where(xr.DataArray(exact | bracket, dims='time', coords={'time': target_time}))


def build_themis_fac(magnetic_gsm, position_gsm, window_sec=100., gap_sec=0.25, max_orbit_gap_sec=None, singularity_tol=1e-10):
    dt = np.diff(magnetic_gsm.time.values) / np.timedelta64(1, 's')
    if not np.isfinite(window_sec) or window_sec <= 0 or np.any(dt <= 0):
        raise ValueError('背景窓は正の有限値、磁場時刻は単調増加にする。')
    if not np.isfinite(gap_sec) or gap_sec <= 0 or not 0 < singularity_tol < 1:
        raise ValueError('ギャップ閾値・特異点閾値が不正。')
    cadence = float(np.median(dt))
    n_window = max(1, int(window_sec / cadence))  # 参照と同じ標本数ベースの中心窓
    seg_id = np.cumsum(np.r_[False, dt > gap_sec])
    smoothed = []
    # 時間ギャップをまたぐ窓を作らず、全点がそろう窓のみを採用する。
    for segment in np.unique(seg_id):
        block = magnetic_gsm.isel(time=np.flatnonzero(seg_id == segment))
        if block.sizes['time'] < n_window:
            smoothed.append(xr.full_like(block, np.nan, dtype=float))
        else:
            smoothed.append(block.rolling(time=n_window, center=True, min_periods=n_window).mean())
    background = xr.concat(smoothed, dim='time').transpose('time', 'v_dim')
    b = background.values
    bnorm = np.linalg.norm(b, axis=1)
    ez = np.full_like(b, np.nan)
    good_b = np.isfinite(b).all(axis=1) & (bnorm > 0)
    ez[good_b] = b[good_b] / bnorm[good_b, None]
    radial = interpolate_position_direction(position_gsm, background.time, max_gap_sec=max_orbit_gap_sec).values
    tangent = radial - np.einsum('ti,ti->t', radial, ez)[:, None] * ez
    tangent_norm = np.linalg.norm(tangent, axis=1)
    valid = good_b & np.isfinite(radial).all(axis=1) & (tangent_norm > singularity_tol)
    ex = np.full_like(b, np.nan)
    ex[valid] = tangent[valid] / tangent_norm[valid, None]
    ey = np.cross(ez, ex)
    basis = []
    for label, values in zip('xyz', (ex, ey, ez)):
        basis.append(xr.DataArray(
            values, dims=('time', 'gsm_component'),
            coords={'time': background.time.values, 'gsm_component': ['x', 'y', 'z']},
            name=f'da_e_{label}_FAC_inGSM',
            attrs={'coordinate_system': 'GSM', 'units': '1'},
        ))
    validity = xr.DataArray(valid, dims='time', coords={'time': background.time}, name='fac_valid')
    background.attrs.update({'units': 'nT', 'window_samples': n_window, 'median_cadence_sec': cadence})
    return background, basis, validity


da_B_background, _fac_basis, fac_valid = build_themis_fac(
    B16_data_gsm, da_THA_pos_gsm, window_sec=background_time_sec,
    gap_sec=fgm_gap_threshold_sec, max_orbit_gap_sec=orbit_max_gap_sec,
    singularity_tol=angle_singularity_tol,
)
da_e_x_FAC_inGSM, da_e_y_FAC_inGSM, da_e_z_FAC_inGSM = _fac_basis
R_FAC_to_GSM = xr.DataArray(
    np.stack([v.values for v in _fac_basis], axis=-1),
    dims=('time', 'gsm_component', 'fac_component'),
    coords={'time': da_B_background.time.values, 'gsm_component': list('xyz'), 'fac_component': list('xyz')},
    name='R_FAC_to_GSM', attrs={'units': '1', 'description': 'v_GSM = R_FAC_to_GSM @ v_FAC'},
)
R_GSM_to_FAC = R_FAC_to_GSM.transpose('time', 'fac_component', 'gsm_component').rename('R_GSM_to_FAC')
R_GSM_to_FAC.attrs = {'units': '1', 'description': 'v_FAC = R_GSM_to_FAC @ v_GSM'}
_valid_matrices = R_FAC_to_GSM.values[fac_valid.values]
if _valid_matrices.shape[0] == 0:
    raise ValueError('有効なFAC基底がない。FGM・軌道の範囲と背景窓を確認する。')
np.testing.assert_allclose(
    np.swapaxes(_valid_matrices, -1, -2) @ _valid_matrices,
    np.broadcast_to(np.eye(3), _valid_matrices.shape), atol=coordinate_matrix_atol, rtol=0,
)
np.testing.assert_allclose(np.linalg.det(_valid_matrices), 1., atol=coordinate_matrix_atol, rtol=0)
print('valid FAC samples:', int(fac_valid.sum()), '/', fac_valid.size)
print('background window:', da_B_background.attrs)

## θ・ξ：GSM XY平面を基準とする共通定義

FAC-Z（背景磁場方向）の正規化したGSM成分を$(e_X,e_Y,e_Z)$として、
$\theta_{\mathrm{GSM}}=\sin^{-1}(e_Z)$、
$\xi_{\mathrm{GSM}}=(\operatorname{atan2}(e_Y,e_X)+90^\circ)\bmod 360^\circ-90^\circ$と定義する。
θはGSM XY平面からの符号付き仰角（−90–90°、+Z側が正）、
ξはXY投影の方位角（−90°以上270°未満、+Xから+Y方向を正）である。
角度計算には位置を使わない。FAC-X・Yを構成する位置データの扱いは維持する。
XY投影が消える場合はξをNaNとし、図の−90°/270°境界で線を切る。
以前の動径基準の角度とは別の量である。


In [ ]:
# 共通定義: θ=asin(e_Z), ξ=atan2(e_Y, e_X)。単位はdegree。
# θはGSM XY平面からの仰角、ξは+Xから+Y方向を正とする方位角。
# 位置データは角度計算に不要。FAC基底の構成方法は変更しない。
def make_fac_orientation_angles(fac_z_gsm, singularity_tol=1e-10):
    """FAC-ZのGSM XY平面からの仰角θと、+X→+Yの方位角ξ（degree）。"""
    ez_da = fac_z_gsm.transpose('time', 'gsm_component').sel(gsm_component=['x', 'y', 'z'])
    times = ez_da.time.values.astype('datetime64[ns]')
    if times.size == 0 or np.isnat(times).any() or np.any(np.diff(times) <= np.timedelta64(0, 'ns')):
        raise ValueError('FAC時刻は空でなく、NaTを含まず単調増加である必要がある。')
    if not np.isfinite(singularity_tol) or not 0 < singularity_tol < 1:
        raise ValueError('singularity_tolは0より大きく1未満を指定する。')
    raw = np.asarray(ez_da.values, dtype=np.float64)
    norm = np.linalg.norm(raw, axis=1)
    valid = np.isfinite(raw).all(axis=1) & np.isfinite(norm) & (norm > 0)
    ez = np.full_like(raw, np.nan)
    ez[valid] = raw[valid] / norm[valid, None]
    theta = np.rad2deg(np.arcsin(np.clip(ez[:, 2], -1., 1.)))
    # XY投影が消える場合、仰角は±90°だが方位角は定義できない。
    xi_valid = valid & (np.hypot(ez[:, 0], ez[:, 1]) > singularity_tol)
    xi = np.full(theta.shape, np.nan)
    azimuth_deg = np.rad2deg(np.arctan2(ez[xi_valid, 1], ez[xi_valid, 0]))
    xi[xi_valid] = (azimuth_deg + 90.) % 360. - 90.
    # 丸めで上端になった場合も[-90, 270)へ収める。
    xi[xi >= 270.] = -90.
    angles = xr.Dataset(
        {'theta_deg': ('time', theta), 'xi_deg': ('time', xi)},
        coords={'time': ez_da.time.values},
        attrs={
            'coordinate_system': 'GSM',
            'definition_id': 'gsm_xy_elevation_azimuth_m90_270_v1',
            'definition': 'theta=asin(e_z.GSM_Z); xi=(atan2(e_z.GSM_Y, e_z.GSM_X) degrees + 90) mod 360 - 90 degrees',
            'singularity_tol': singularity_tol,
        },
    )
    angles.theta_deg.attrs = {
        'units': 'degree', 'valid_range': [-90., 90.],
        'description': 'Signed elevation of FAC-Z above GSM XY plane; positive toward +Z',
    }
    angles.xi_deg.attrs = {
        'units': 'degree', 'valid_range': [-90., 270.],
        'description': 'Azimuth of FAC-Z projected onto GSM XY plane in [-90, 270) degrees; positive from +X toward +Y',
    }
    return angles


fac_orientation_angles = make_fac_orientation_angles(
    da_e_z_FAC_inGSM, singularity_tol=angle_singularity_tol,
)
print(fac_orientation_angles)


## 描画関数

Arase版の描画関数を基に、探査機名とパネル番号だけをTHEMIS-A用に変更した。
極座標版・直交座標版とも同じ図サイズと時系列レイアウトを使う。

θ・ξのデータ欠損は、両側の有効な角度を結ぶ**点線**で表示する（表示用線形補間）。
これは参照解析の`R_GSM_to_FAC.interp(time=...)`による**行列成分の補間とは別**で、
θ・ξ・FAC基底・スナップショットの選択結果は変更しない。
ξは最短の円周方向でつなぎ、−90°/270°で線を切る。ξだけが未定義になる幾何学的特異点、
先頭・末尾の欠損はつながない。補間する端点は表示区間内に限定する。


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
mpl.rcParams['font.size'] = 15

def fac_basis_gsm_at(it):
    basis = xr.align(da_e_x_FAC_inGSM, da_e_y_FAC_inGSM, da_e_z_FAC_inGSM, join='exact')
    vectors = np.stack([
        da.isel(time=it).sel(gsm_component=['x', 'y', 'z']).values for da in basis
    ])  # (FAC軸, GSM成分)。配列内の座標ラベルは既存の小文字を維持。
    if not np.isfinite(vectors).all():
        raise ValueError(f'FAC基底に非有限値がある: time index={it}')
    return vectors


def setup_polar_ax(ax, horizontal, vertical):
    ax.set_title(f'GSM ({horizontal}, {vertical}) plane')
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(1)
    ax.set_thetagrids(np.arange(0, 360, 45))
    minor_deg = np.arange(0, 360, 15)
    ax.set_xticks(np.deg2rad(minor_deg[minor_deg % 45 != 0]), minor=True)
    ax.tick_params(axis='x', which='minor', labelbottom=False)
    ax.set_ylim(0, 1.05)
    ax.set_rticks([0.25, 0.5, 0.75, 1.0])
    ax.set_rlabel_position(135)
    ax.grid(True, which='major')
    ax.xaxis.grid(True, which='minor', linestyle=':', linewidth=0.6, alpha=0.6)


def cart_to_polar(u, v):
    return np.arctan2(v, u), np.hypot(u, v)


def plot_polar_vector(ax, u, v, color, label):
    theta, r = cart_to_polar(u, v)
    ax.plot([theta, theta], [0, r], color=color, lw=2, label=label)
    ax.plot(theta, r, 'o', color=color, ms=5)


def setup_ax(ax, horizontal, vertical):
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_xlabel(f'GSM-{horizontal}')
    ax.set_ylabel(f'GSM-{vertical}')
    ax.minorticks_on()
    ax.grid(True, which='both', alpha=0.3)
    ax.set_title(f'GSM ({horizontal}, {vertical}) plane')


def angle_line_with_breaks(times, values, wrapped=False):
    """ξの−90°/270°境界と時間欠損にNaNを挿入し、両側の有効点を保持する。"""
    times = np.asarray(times).astype('datetime64[ns]')
    values = np.asarray(values, dtype=float)
    if times.size < 2:
        return times, values
    dt = np.diff(times) / np.timedelta64(1, 's')
    breaks = dt > 3 * np.median(dt)
    if wrapped:
        breaks |= np.abs(np.diff(values)) > 180.
    indices = np.flatnonzero(breaks) + 1
    return np.insert(times, indices, times[indices]), np.insert(values, indices, np.nan)


def plot_angle_gap_bridges(ax, times, values, color, wrapped=False, forbidden=None):
    """欠損両側の角度だけから作る表示用の線形補間。元データは変更しない。

    ξは最短の円周方向で補間し、−90°/270°の境界で描画線を切る。
    forbidden=Trueの点を含む区間（ξの幾何学的特異点など）は接続しない。
    先頭・末尾の外挿も行わない。通常の隣接サンプルは実線側で描く。
    """
    times = np.asarray(times).astype('datetime64[ns]')
    values = np.asarray(values, dtype=float)
    finite_indices = np.flatnonzero(np.isfinite(values))
    if times.size < 2 or finite_indices.size < 2:
        return []
    forbidden = np.zeros(times.size, dtype=bool) if forbidden is None else np.asarray(forbidden, dtype=bool)
    if forbidden.shape != values.shape:
        raise ValueError('forbiddenの形状は角度配列と一致させる。')
    cadence_sec = float(np.median(np.diff(times) / np.timedelta64(1, 's')))
    left, right = finite_indices[:-1], finite_indices[1:]
    gap = (right > left + 1) | ((times[right] - times[left]) / np.timedelta64(1, 's') > 3 * cadence_sec)
    artists = []
    for i, j in zip(left[gap], right[gap]):
        if forbidden[i:j + 1].any():
            continue
        fraction = np.linspace(0., 1., 65)
        delta = values[j] - values[i]
        if wrapped:
            delta = (delta + 180.) % 360. - 180.
        interpolated = values[i] + fraction * delta
        if wrapped:
            interpolated = (interpolated + 90.) % 360. - 90.
        span_ns = (times[j] - times[i]) / np.timedelta64(1, 'ns')
        bridge_times = times[i] + np.rint(fraction * span_ns).astype('timedelta64[ns]')
        if wrapped:
            cut = np.flatnonzero(np.abs(np.diff(interpolated)) > 180.) + 1
            bridge_times = np.insert(bridge_times, cut, bridge_times[cut])
            interpolated = np.insert(interpolated, cut, np.nan)
        artist, = ax.plot(
            bridge_times, interpolated, color=color, linestyle=':', lw=1.5,
            label='Gap bridge (linear)',
        )
        artists.append(artist)
    return artists


def add_orientation_time_series(fig, gridspec, snapshot_time, time_range=None, angle_limits=None):
    angles = fac_orientation_angles
    if angles.attrs.get('definition_id') != 'gsm_xy_elevation_azimuth_m90_270_v1':
        raise ValueError('角度の定義が旧版。GSM XY基準の角度計算セルを再実行する。')
    # 古い角度計算結果を誤って別のFAC時系列に対応させない。
    if not np.array_equal(angles.time.values, da_e_z_FAC_inGSM.time.values):
        raise ValueError('角度とFACの時刻列が異なる。角度計算セルを再実行する。')
    if time_range is not None:
        angles = angles.sel(time=slice(*time_range))
        if angles.sizes['time'] == 0:
            raise ValueError('指定時間範囲に角度データがない。')
    ax_theta = fig.add_subplot(gridspec[1, :])
    ax_xi = fig.add_subplot(gridspec[2, :], sharex=ax_theta)
    for ax, key, color, panel, ylabel, ylim in (
        (ax_theta, 'theta_deg', 'tab:blue', '(d)', r'$\theta$ [deg]', (-90, 90)),
        (ax_xi, 'xi_deg', 'tab:orange', '(e)', r'$\xi$ [deg]', (-90, 270)),
    ):
        t, values = angle_line_with_breaks(angles.time.values, angles[key].values, wrapped=(key == 'xi_deg'))
        ax.plot(t, values, color=color, lw=1.2)
        # ξだけがNaNでθが有限なら、FAC-ZのXY投影が消える特異点。
        # その区間は、単なるデータ欠損として橋渡ししない。
        forbidden = None
        if key == 'xi_deg':
            forbidden = np.isfinite(angles.theta_deg.values) & ~np.isfinite(angles.xi_deg.values)
        bridges = plot_angle_gap_bridges(
            ax, angles.time.values, angles[key].values, color,
            wrapped=(key == 'xi_deg'), forbidden=forbidden,
        )
        if bridges:
            ax.legend(handles=[bridges[0]], loc='best', fontsize=15)
        ax.axvline(snapshot_time, color='red', linestyle='--', lw=1.5)
        ax.set_ylabel(ylabel)
        ax.set_ylim(*(angle_limits[key] if angle_limits is not None else ylim))
        ax.minorticks_on()
        ax.grid(True, which='both', alpha=0.3)
        ax.text(0.01, 0.96, panel, transform=ax.transAxes, ha='left', va='top', fontweight='bold')
    if angle_limits is None:
        ax_theta.set_yticks([-90, -45, 0, 45, 90])
        ax_xi.set_yticks([-90, 0, 90, 180, 270])
    else:
        for ax in (ax_theta, ax_xi):
            ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=5))
    if time_range is not None:
        ax_xi.set_xlim(*time_range)
    elif angles.sizes['time'] > 1:
        ax_xi.set_xlim(angles.time.values[0], angles.time.values[-1])
    locator = mdates.AutoDateLocator(minticks=4, maxticks=10)
    ax_xi.xaxis.set_major_locator(locator)
    ax_xi.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax_theta.tick_params(axis='x', labelbottom=False)
    ax_xi.set_xlabel('Time [UTC]')
    return ax_theta, ax_xi


def make_fac_gsm_figure(it, style='polar', time_range=None, angle_limits=None):
    """上段(a)-(c): FAC基底、中段(d): θ、下段(e): ξ。"""
    if style not in ('polar', 'cartesian'):
        raise ValueError("styleは'polar'または'cartesian'を指定する。")
    vectors = fac_basis_gsm_at(it)
    fig = plt.figure(figsize=(16, 11), layout='constrained')
    fig.get_layout_engine().set(h_pad=0.12, w_pad=0.08)
    gs = fig.add_gridspec(3, 3, height_ratios=[2.8, 1, 1], hspace=0.10)
    snapshot_time = da_e_x_FAC_inGSM.time.values[it]
    for col, ((i, j), panel) in enumerate(zip(((0, 1), (0, 2), (1, 2)), ('(a)', '(b)', '(c)'))):
        ax = fig.add_subplot(gs[0, col], projection='polar' if style == 'polar' else None)
        horizontal, vertical = 'XYZ'[i], 'XYZ'[j]
        if style == 'polar':
            setup_polar_ax(ax, horizontal, vertical)
            plot_polar_vector(ax, 1, 0, 'k', f'GSM-{horizontal}')
            plot_polar_vector(ax, 0, 1, 'gray', f'GSM-{vertical}')
            for vector, color, label in zip(vectors, ('r', 'b', 'g'), ('FAC-x', 'FAC-y', 'FAC-z')):
                plot_polar_vector(ax, vector[i], vector[j], color, label)
        else:
            setup_ax(ax, horizontal, vertical)
            for u, v, color, label in ((1, 0, 'k', f'GSM-{horizontal}'), (0, 1, 'gray', f'GSM-{vertical}')):
                ax.quiver(0, 0, u, v, angles='xy', scale_units='xy', scale=1, color=color, label=label)
            for vector, color, label in zip(vectors, ('r', 'b', 'g'), ('FAC-x', 'FAC-y', 'FAC-z')):
                ax.quiver(0, 0, vector[i], vector[j], angles='xy', scale_units='xy', scale=1, color=color, label=label)
        ax.text(-0.08, 1.10, panel, transform=ax.transAxes, ha='left', va='top', fontweight='bold')
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=15)
    add_orientation_time_series(fig, gs, snapshot_time, time_range=time_range, angle_limits=angle_limits)
    time_label = np.datetime_as_string(snapshot_time, unit='ms')
    fig.suptitle(f'{spacecraft_label} | FAC unit vectors in GSM | {time_label} UTC', fontsize=20)
    return fig


def save_fac_gsm_frame(it, save_dir, style, time_range=None, angle_limits=None):
    fig = make_fac_gsm_figure(it, style=style, time_range=time_range, angle_limits=angle_limits)
    time_label = np.datetime_as_string(da_e_x_FAC_inGSM.time.values[it], unit='ms')
    time_stamp = time_label.replace('-', '').replace(':', '').replace('T', '_').replace('.', '_')
    os.makedirs(save_dir, exist_ok=True)
    fname = os.path.join(save_dir, f'{spacecraft_label}_fac_in_gsm_{style}_{time_stamp}UTC.png')
    try:
        fig.savefig(fname, dpi=150)
    finally:
        plt.close(fig)
    return fname


def plot_fac_gsm_frame_polar(it, frame_idx, save_dir):
    return save_fac_gsm_frame(it, save_dir, style='polar')


def select_minute_frames(times, tolerance_sec=None, target_start=None, target_end=None):
    """毎分00秒に最も近いサンプルを選ぶ。戻り値のoffsetは実時刻−00秒（秒）。"""
    times = np.asarray(times).astype('datetime64[ns]')
    if times.ndim != 1 or times.size == 0 or np.isnat(times).any():
        raise ValueError('時刻は空でない1次元配列で、NaTを含まない必要がある。')
    dt_ns = np.diff(times.astype(np.int64))
    if np.any(dt_ns <= 0):
        raise ValueError('時刻は単調増加である必要がある。')
    # 既定許容差はサンプル間隔の中央値。長い欠損区間をまたいで選ばない。
    if tolerance_sec is None:
        tolerance_sec = float(np.median(dt_ns) / 1e9) if dt_ns.size else 0.
    if not np.isfinite(tolerance_sec) or tolerance_sec < 0:
        raise ValueError('tolerance_secは非負の有限値を指定する。')
    first_minute = times[0].astype('datetime64[m]').astype('datetime64[ns]')
    if first_minute < times[0]:
        first_minute += np.timedelta64(1, 'm')
    if target_start is not None:
        first_minute = np.datetime64(target_start, 'ns')
    last_time = times[-1] if target_end is None else np.datetime64(target_end, 'ns')
    if np.isnat(first_minute) or np.isnat(last_time) or ((target_start is not None or target_end is not None) and last_time < first_minute):
        raise ValueError('毎分00秒の選択対象範囲が不正。')
    targets = np.arange(first_minute, last_time + np.timedelta64(1, 'ns'),
                        np.timedelta64(1, 'm'))
    right = np.clip(np.searchsorted(times, targets), 0, times.size - 1)
    left = np.maximum(right - 1, 0)
    # 同距離なら前のサンプルを選ぶ。
    indices = np.where(np.abs(times[left] - targets) <= np.abs(times[right] - targets), left, right)
    offsets = (times[indices] - targets) / np.timedelta64(1, 's')
    keep = np.abs(offsets) <= tolerance_sec
    return targets, indices, offsets, keep, tolerance_sec

In [ ]:
def angle_limits_for_window(angles, time_start, time_end, padding_fraction=0.05, min_padding_deg=0.5):
    """区間全体の有限値から余白付き範囲を決める。全フレームで共通に使う。"""
    if padding_fraction < 0 or min_padding_deg <= 0:
        raise ValueError('余白率は非負、最小余白は正にする。')
    selected = angles.sel(time=slice(time_start, time_end))
    limits = {}
    for key, physical_range in (('theta_deg', (-90., 90.)), ('xi_deg', (-90., 270.))):
        values = selected[key].values
        values = values[np.isfinite(values)]
        if values.size == 0:
            raise ValueError(f'{key}: 指定区間に有効値がない。FACデータを確認する。')
        low, high = float(values.min()), float(values.max())
        pad = max((high - low) * padding_fraction, min_padding_deg)
        limits[key] = (max(physical_range[0], low - pad), min(physical_range[1], high + pad))
    return limits

# 時間範囲内の有効値から共通の縦軸範囲を作る。
range_angle_limits = angle_limits_for_window(fac_orientation_angles, range_start, range_end)
print('angle limits [deg]:', range_angle_limits)

## 1枚のプレビュー（任意）

実データを読み込んだ後、このセルで先頭の有効な毎分00秒付近を確認できる。
ここでは表示のみで保存しない。全フレーム保存は最後のセルを実行する。

In [ ]:
# 対象区間の毎分00秒を列挙。実サンプルは区間内に限定する。
if range_end <= range_start or range_plot_style not in ('polar', 'cartesian'):
    raise ValueError('対象時間範囲または描画形式が不正。')
range_all_times = da_e_x_FAC_inGSM.time.values.astype('datetime64[ns]')
range_global_indices = np.flatnonzero((range_all_times >= range_start) & (range_all_times <= range_end))
if range_global_indices.size == 0:
    raise ValueError('対象区間にFAC時刻がない。')
range_targets, range_candidates, range_offsets, range_keep, range_used_tolerance_sec = select_minute_frames(
    range_all_times[range_global_indices], tolerance_sec=range_minute_tolerance_sec,
    target_start=range_start, target_end=range_end,
)
# 無効なFAC基底で近傍時刻を代用しない。
range_keep &= fac_valid.values[range_global_indices[range_candidates]]
range_frame_indices = range_global_indices[range_candidates[range_keep]]
range_frame_selection = xr.Dataset({
    'target_time': ('frame', range_targets[range_keep]),
    'sample_time': ('frame', range_all_times[range_frame_indices]),
    'time_offset_sec': ('frame', range_offsets[range_keep]),
    'sample_index': ('frame', range_frame_indices),
})
range_frame_selection.time_offset_sec.attrs['units'] = 's'
print('frames:', range_frame_indices.size, '/', range_targets.size,
      '| skipped:', int((~range_keep).sum()), '| tolerance [s]:', range_used_tolerance_sec)
if range_frame_indices.size == 0:
    raise ValueError('対象区間の毎分00秒に対応する有効なFAC基底がない。')
print(range_frame_selection)

# プレビューを表示する場合だけコメントを外す。
# fig = make_fac_gsm_figure(int(range_frame_indices[0]), style=range_plot_style,
#                           time_range=(range_start, range_end), angle_limits=range_angle_limits)
# plt.show()
# plt.close(fig)

## 22:25–23:15 UTCの毎分00秒を保存

最大51枚（両端を含む）。別フォルダへ逐次保存し、既存のArase出力には触れない。
ファイル名と赤破線は、00秒の目標時刻ではなく実際に使ったサンプル時刻を示す。

In [ ]:
range_folder_label = (
    f"FAC_in_GSM_{np.datetime_as_string(range_start, unit='D').replace('-', '')}_"
    f"{np.datetime_as_string(range_start, unit='m')[11:].replace(':', '')}-"
    f"{np.datetime_as_string(range_end, unit='m')[11:].replace(':', '')}_UTC"
)
range_save_dir = os.path.join(
    f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_{probe.upper()}/2022-09-01_third/2230-2330/coordinate',
    range_folder_label,
)
print('save directory:', range_save_dir)
os.makedirs(range_save_dir, exist_ok=True)
for it in tqdm(range_frame_indices, desc=f'{spacecraft_label} 22:25–23:15 UTC'):
    save_fac_gsm_frame(
        int(it), range_save_dir, range_plot_style,
        time_range=(range_start, range_end), angle_limits=range_angle_limits,
    )